In [ ]:
import pandas as pd
import os, sys
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

ruta_raiz = os.path.abspath('..')
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

%load_ext autoreload
%autoreload 2
from Data_preprocesing.IQRCapper import IQRCapper



X_train = pd.read_csv("../Train_test_split/X_train.csv")
y_train = pd.read_csv("../Train_test_split/y_train.csv").values.ravel()
X_test = pd.read_csv("../Train_test_split/X_test.csv")
y_test = pd.read_csv("../Train_test_split/y_test.csv").values.ravel()

target_folder = '../comparations'
os.makedirs(target_folder, exist_ok=True) 


def save_experiment(model_name, imbalance_method, accuracy, precision, recall, f1, roc_auc):
    new_result = {
        'Model': [model_name],
        'Imbalance_Method': [imbalance_method],
        'Accuracy': [round(accuracy, 4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall, 4)],
        'F1_Score': [round(f1, 4)],
        'ROC_AUC': [round(roc_auc, 4)]
    }
    df_new = pd.DataFrame(new_result)
    
    
    csv_file = f'{target_folder}/imbalance_results.csv'
    
    if os.path.exists(csv_file):
        df_new.to_csv(csv_file, mode='a', header=False, index=False)
    else:
        df_new.to_csv(csv_file, index=False)
        
    print(f"✅ Success! Results for {model_name} + {imbalance_method} saved in the '{target_folder}' folder.")

C:\Users\alvarodela.herran\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\alvarodela.herran\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


BaggingClassifier

In [7]:
import sys
sys.path.append("C:\\Users\\alvarodela.herran\\Machine_learning\\Adv_ML\\Data_preprocesing")
from IQRCapper import IQRCapper

from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

imbalance_methods = {
    "Baseline": None,
    "RandomUnderSampler":     RandomUnderSampler(random_state=42),
    "TomekLinks":             TomekLinks(),
    "RandomOverSampler":      RandomOverSampler(random_state=42),
    "SMOTE":                  SMOTE(random_state=42),
    "ADASYN":                 ADASYN(random_state=42),
    "SMOTETomek":             SMOTETomek(random_state=42),
    "SMOTEENN":               SMOTEENN(random_state=42),
}

bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

for method_name, sampler in imbalance_methods.items():
    print(f"\n--- {method_name} ---")

    if sampler is None:
        pipe = ImbPipeline([
            ('capping_outliers', IQRCapper(factor=1.5)),
            ('imputacion', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('classifier', bagging)])
    else:
        pipe = ImbPipeline([
            ('capping_outliers', IQRCapper(factor=1.5)),
            ('imputacion', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('sampler', sampler),
            ('classifier', bagging)])

    pipe.fit(X_train, y_train)

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    accuracy  = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall    = recall_score(y_test, y_pred, average='weighted')
    f1        = f1_score(y_test, y_pred, average='weighted')
    roc_auc   = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

    print(f"Accuracy: {accuracy:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

    save_experiment(
        model_name="BaggingClassifier v2 (DecisionTree)",
        imbalance_method=method_name,
        accuracy=accuracy,
        precision=precision,
        recall=recall,
        f1=f1,
        roc_auc=roc_auc
    )


--- Baseline ---
Accuracy: 0.7508 | F1: 0.7481 | ROC-AUC: 0.8963
✅ Success! Results for BaggingClassifier v2 (DecisionTree) + Baseline saved in the '../comparations' folder.

--- RandomUnderSampler ---
Accuracy: 0.7286 | F1: 0.7319 | ROC-AUC: 0.8911
✅ Success! Results for BaggingClassifier v2 (DecisionTree) + RandomUnderSampler saved in the '../comparations' folder.

--- TomekLinks ---
Accuracy: 0.7528 | F1: 0.7497 | ROC-AUC: 0.8958
✅ Success! Results for BaggingClassifier v2 (DecisionTree) + TomekLinks saved in the '../comparations' folder.

--- RandomOverSampler ---
Accuracy: 0.7454 | F1: 0.7458 | ROC-AUC: 0.8957
✅ Success! Results for BaggingClassifier v2 (DecisionTree) + RandomOverSampler saved in the '../comparations' folder.

--- SMOTE ---
Accuracy: 0.7370 | F1: 0.7387 | ROC-AUC: 0.8934
✅ Success! Results for BaggingClassifier v2 (DecisionTree) + SMOTE saved in the '../comparations' folder.

--- ADASYN ---
Accuracy: 0.7374 | F1: 0.7393 | ROC-AUC: 0.8935
✅ Success! Results for Ba

Hyperparameter Optuna for BaggingClassifier

In [6]:
import sys
sys.path.append("C:\\Users\\alvarodela.herran\\Machine_learning\\Adv_ML\\Data_preprocesing")
from IQRCapper import IQRCapper

import optuna
from imblearn.under_sampling import TomekLinks
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # Hyperparameters for BaggingClassifier
    n_estimators      = trial.suggest_int('n_estimators', 50, 500)
    max_samples       = trial.suggest_float('max_samples', 0.5, 1.0)
    max_features      = trial.suggest_float('max_features', 0.5, 1.0)

    # Hyperparameters for base DecisionTree
    max_depth         = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)

    pipe = ImbPipeline([
        ('capping_outliers', IQRCapper(factor=1.5)),
        ('imputacion',       SimpleImputer(strategy='mean')),
        ('scaler',           StandardScaler()),
        ('sampler',          TomekLinks()),
        ('classifier',       BaggingClassifier(
            estimator=DecisionTreeClassifier(
                max_depth=max_depth,
                min_samples_split=min_samples_split
            ),
            n_estimators=n_estimators,
            max_samples=max_samples,
            max_features=max_features,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        ))
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    return f1_score(y_test, y_pred, average='weighted')

# Create study and optimize with 50 trials
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"\nBest F1: {study.best_value:.4f}")
print(f"Best hyperparameters: {study.best_params}")


Best F1: 0.7527
Best hyperparameters: {'n_estimators': 499, 'max_samples': 0.7617356445426349, 'max_features': 0.9557479213602055, 'max_depth': 13, 'min_samples_split': 20}


In [8]:
import optuna.visualization as ov

ov.plot_param_importances(study).show()
ov.plot_optimization_history(study).show()
ov.plot_slice(study).show()

In [9]:
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

best = study.best_params

pipe_final = ImbPipeline([
    ('capping_outliers', IQRCapper(factor=1.5)),
    ('imputacion',       SimpleImputer(strategy='mean')),
    ('scaler',           StandardScaler()),
    ('sampler',          TomekLinks()),
    ('classifier',       BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=best['max_depth'],
            min_samples_split=best['min_samples_split']
        ),
        n_estimators=best['n_estimators'],
        max_samples=best['max_samples'],
        max_features=best['max_features'],
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    ))
])

pipe_final.fit(X_train, y_train)
y_pred  = pipe_final.predict(X_test)
y_proba = pipe_final.predict_proba(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')
roc_auc   = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

save_experiment(
    model_name="BaggingClassifier v2 Optuna (Final)",
    imbalance_method="TomekLinks",
    accuracy=accuracy,
    precision=precision,
    recall=recall,
    f1=f1,
    roc_auc=roc_auc
)

Accuracy:  0.7555
Precision: 0.7544
Recall:    0.7555
F1:        0.7527
ROC-AUC:   0.8994
✅ Success! Results for BaggingClassifier v2 Optuna (Final) + TomekLinks saved in the '../comparations' folder.
